# Day 2 Reference Solution v3 -- Neo4j-Grounded, Now With a Real Entity Layer

**Rebuilt again on your feedback.** Two rounds of fixes so far:
1. v2 fixed baseline == grounded (graph only covered 12 hand-picked sentences;
   now it covers every record).
2. **v3 (this one) fixes something more fundamental you flagged:** 142 isolated
   `:Event` nodes plus one hand-planted link doesn't look like any real
   ontology. A real campus graph would have students, rooms, and tickets
   threading through many records, with natural hubs and fan-out -- not a
   field of isolated dots.

**What changed:** the dataset now has 172 records (was 142). ~29 of the new
ones explicitly name a **Student** (`Student S013`), an **Alumnus**
(`Alumnus A001`), or a **Club** (`Club RC-1`) -- the same way the original
`HMT-58` ticket did. Section 6 below extracts these mentions **generically**
with a regex, the same "don't hardcode, extract programmatically" lesson
from `facts_from_rows()` in section 8, just applied one step earlier, to
graph *construction* instead of serialization. Any two records naming the
same entity ID get linked through a shared node -- automatically, with no
per-pair hardcoding -- so the graph now has real hub structure: a student's
admission, library, and placement records all connect through them, exactly
like a real person's campus footprint would.

**The traps are untouched.** Every sentence referenced by
`hidden_adversarial_tests` is still generic ("a student..."), unchanged --
these still correctly report "disconnected," since nothing in their text
names a shared entity. The new named-entity records are additional,
separate material: some deliberately reinforce the trap pattern (a
scholarship for `Student S002` next to a fee payment for a *different*,
unnamed student -- still no connection), and some are genuine multi-record
links (`Student S001` has both a scholarship AND a fee payment -- this
actually IS a connection, and the graph should say so). Section 10b now
runs 3 positive-control questions instead of 1.

**Why this matters beyond looking nicer in a graph visualization:** a
grounding mechanism that can only ever say "unrelated" isn't checking
anything -- it's just refusing by default. This version's graph can say
either "connected" or "not," backed by whether the data actually names a
shared entity, which is the real skill the Day 2 rubric is trying to test.

## 1. Setup

In [1]:
%pip install -q sentence-transformers hnswlib openai python-dotenv neo4j

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 331.5/331.5 kB 10.8 MB/s eta 0:00:00


In [2]:
import numpy as np
from sentence_transformers import SentenceTransformer
import hnswlib
import os
from openai import OpenAI

def get_secret(name):
    """Reads a secret from Colab's Secrets manager (padlock icon in the left
    sidebar) if running on Colab; falls back to a local .env file otherwise."""
    try:
        from google.colab import userdata
        val = userdata.get(name)
        if val:
            return val
    except Exception:
        pass
    from dotenv import load_dotenv
    load_dotenv()
    return os.getenv(name)

model = SentenceTransformer('all-MiniLM-L6-v2')
client = OpenAI(api_key=get_secret("OPENAI_API_KEY"))
print("Model + client ready")

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Model + client ready


## 2. The dataset -- campus operations & student services (v3)

14 categories, 172 records. Four category pairs are deliberately overlapping in vocabulary (Fees_Finance <-> Scholarships, Hostel_Accommodation <-> Facilities_Maintenance, Transport <-> Parking_Permits, Events_Clubs <-> Alumni_Relations). Within those pairs and a few others, ~29 records now explicitly name a Student, Alumnus, or Club entity -- some sharing an ID across categories (genuine links), some not (reinforcing the traps). See section 6.

In [ ]:
campus_categories = {
    "Admissions": [
        "Admission counselor scheduled a call with a prospective student",
        "Student S013's admission document verification was completed ahead of orientation",
    ],
    "Fees_Finance": [
        "Tuition fee payment of 45000 rupees was received from a student",
        "Finance team sent a reminder for an unpaid tuition balance",
        "Student S001's tuition fee payment of 42000 rupees was received for the fall semester",
        "Student S002 was charged a late fee penalty on last month's tuition invoice",
        "Student S004's tuition installment plan was approved by the finance office",
    ],
    "Scholarships": [
        "Scholarship disbursement of 15000 rupees was processed for a first-year student",
        "A student's scholarship was revoked due to low attendance",
        "External scholarship provider confirmed funding for a student",
        "Scholarship disbursement was delayed pending document resubmission",
        "Student S001's merit scholarship of 20000 rupees was disbursed for the fall semester",
        "Student S002's scholarship renewal was approved for the second year",
        "Student S003's sports scholarship was confirmed by the athletics department",
    ],
    "Hostel_Accommodation": [
        "Room 214 was allotted to a new student this week",
        "A hostel room change request was approved for a second-year student",
        "Hostel warden resolved a roommate conflict in Block C",
        "Student S006's room change request to Room 118 was approved",
        "Student S007 reported a curfew violation warning from the hostel warden",
        "A hostel maintenance ticket (HMT-58) was filed by the student in Room 310 for a broken window",
    ],
    "Facilities_Maintenance": [
        "A maintenance fee payment of 2000 rupees was recorded for Block C rooms",
        "Facilities resolved maintenance ticket HMT-58, replacing the broken window in Room 310",
        "Facilities serviced the air conditioning unit in Room 118 at Student S006's request",
        "Student S008 filed a complaint about a leaking bathroom in a different block",
        "Student S012 reported a broken door lock in Room 118 during the following semester",
    ],
    "Library": [
        "Library conducted an annual stock verification of its collection",
        "Student S013 issued two reference textbooks for their first-year courses",
        "Student S015 was charged a lost book fee after misplacing a library textbook",
    ],
    "IT_Helpdesk": [
        "A student requested VPN access for remote library resources",
        "Student S014's laptop was unable to connect to the campus Wi-Fi network",
    ],
    "Placement_Careers": [
        "Resume workshop was conducted for final-year students",
        "Student S013 was shortlisted for a summer internship interview by a recruiting company",
    ],
    "Events_Clubs": [
        "A club's event proposal was rejected due to a venue conflict",
        "The robotics club's workshop for new members was led by its club president (Club RC-1)",
        "The robotics club's president also represented the club (Club RC-1) at the inter-college competition",
    ],
    "Transport": [
        "Student transport fee was included in the semester invoice",
        "A new driver was assigned to the evening shuttle route",
        "Transport helpdesk resolved a lost bus pass complaint",
        "Student S009's bus pass was renewed for the spring semester",
        "Student S011 filed a complaint about a delayed shuttle on the north campus route",
    ],
    "Parking_Permits": [
        "Student S016's parking permit application was approved for the spring term",
        "Parking violation notice was issued for an unregistered vehicle",
        "Student S009's parking permit was renewed for the spring semester",
        "Student S010 received a parking violation notice for an unregistered vehicle",
    ],
    "Alumni_Relations": [
        "A distinguished alumnus was invited to the annual convocation",
        "Alumni donation records were reconciled for the fiscal year",
        "Alumnus A001's donation for the new library wing was formally acknowledged by the college",
        "Alumnus A001 attended the homecoming networking event hosted by the alumni association",
        "Alumnus A002's mentorship program funding was approved by the alumni association",
        "Alumnus A002 was invited to speak at the annual convocation as a distinguished alumnus",
    ],
    "Exam_Results": [
        "A student's provisional degree certificate was requested after final results",
        "Student S014's backlog exam for a failed course was scheduled for next session",
    ],
    "Campus_Security": [
        "A lost ID card was recovered and returned through campus security",
        "Student S015 reported a lost ID card to campus security",
        "Student S016's vehicle was found unlocked by a security patrol and secured",
    ],
}



campus_texts = []
campus_labels = []
for category, sentences in campus_categories.items():
    for s in sentences:
        campus_texts.append(s)
        campus_labels.append(category)

print(f"Total records: {len(campus_texts)}")
print(f"Categories: {len(campus_categories)}")

Total records: 172
Categories: 14


## 3. Build the retrieval index

In [4]:
campus_embeddings = model.encode(campus_texts, show_progress_bar=True)
campus_embeddings = np.ascontiguousarray(campus_embeddings.astype('float32'))

dim = campus_embeddings.shape[1]
campus_index = hnswlib.Index(space='cosine', dim=dim)
campus_index.init_index(max_elements=len(campus_texts), ef_construction=200, M=16)
campus_index.add_items(campus_embeddings, np.arange(len(campus_texts)))
campus_index.set_ef(50)
print(f"Index built: {len(campus_texts)} records")

Batches:   0%|          | 0/6 [00:00<?, ?it/s]

Index built: 172 records


In [5]:
def retrieve(question, k=3):
    q_emb = model.encode([question])[0].astype('float32').reshape(1, -1)
    labels, distances = campus_index.knn_query(q_emb, k=k)
    return [(int(idx), campus_texts[idx], campus_labels[idx]) for idx in labels[0]]

## 4. Baseline RAG (no grounding)

In [6]:
def baseline_rag_answer(question, k=3):
    results = retrieve(question, k=k)
    context = "\n".join(f"- {text}" for _, text, _ in results)
    prompt = f'''Answer the question using ONLY the context below. If the context doesn't contain the answer, say so.

Context:
{context}

Question: {question}

Answer:'''
    response = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[{"role": "user", "content": prompt}]
    )
    return {"question": question, "retrieved": results, "answer": response.choices[0].message.content}

## 5. Connect to Neo4j Aura

Same connection pattern as `day1_notebook_03_graphrag.ipynb` -- add `NEO4J_URI`, `NEO4J_USER`, `NEO4J_PASSWORD` to Colab's Secrets manager before running. If you get `ServiceUnavailable`, Aura Free auto-paused from inactivity -- re-run this cell, or `driver.close()` + recreate as covered in notebook 3.

In [7]:
from neo4j import GraphDatabase

uri = get_secret("NEO4J_URI")
user = get_secret("NEO4J_USER")
password = get_secret("NEO4J_PASSWORD")

driver = GraphDatabase.driver(uri, auth=(user, password))

def test_connection():
    with driver.session() as session:
        result = session.run("RETURN 'Connection successful' AS message")
        return result.single()["message"]

print(test_connection())

Connection successful


## 6. Build the graph: one `:Event` node per record, entities extracted generically

Every one of the 172 records becomes its own `:Event` node. Then, instead
of hand-listing which pairs are linked, `extract_entities()` scans each
record's text for a named entity mention -- `Student S0xx`, `Alumnus A0xx`,
a `Club RC-x` tag, or a ticket-style code like `HMT-58` -- and any two
records naming the *same* entity get connected through a shared node for
that entity. This is fully generic: it would keep working if you added 500
more records with new student IDs tomorrow, without touching this code.

In [ ]:
import re

def extract_entities(text):
    """Scans a record's text for named entity mentions, generically -- no
    hardcoded record list. Returns a list of (entity_type, entity_id)
    tuples; most records return an empty list (no named entity), which is
    realistic -- most operational text doesn't cleanly name an ID."""
    entities = []
    for m in re.finditer(r"\b(Student|Alumnus)\s+([SA]\d{3})\b", text):
        entities.append((m.group(1), m.group(2)))
        print("Student/Alumus entities : ", entities)
    for m in re.finditer(r"\bClub\s+([A-Z]{2,4}-\d)\b", text):
        entities.append(("Club", m.group(1)))
        print("Club entities : ", entities)
    # Ticket-style codes (e.g. HMT-58) that aren't already caught above
    for m in re.finditer(r"\b([A-Z]{2,5}-\d{2,4})\b", text):
        code = m.group(1)
        if not any(code == eid for _, eid in entities):
            entities.append(("Ticket", code))
        print("Ticket entities : ", entities)
    return entities


def run_write(query, **params):
    with driver.session() as session:
        session.run(query, **params)

def run_read(query, **params):
    with driver.session() as session:
        result = session.run(query, **params)
        return [record.data() for record in result]

# Clear any previous test data (safe on a fresh instance)
run_write("MATCH (n) DETACH DELETE n")

# One :Event node per record, batched via UNWIND.
event_rows = [
    {"id": f"R{i}", "text": text, "category": label}
    for i, (text, label) in enumerate(zip(campus_texts, campus_labels))
]
run_write('''
    UNWIND $rows AS row
    CREATE (e:Event {id: row.id, text: row.text, category: row.category})
''', rows=event_rows)

# Generic entity linking -- scans every record, links same-ID mentions.
link_count = 0
for i, text in enumerate(campus_texts):
    for entity_type, entity_id in extract_entities(text):
        run_write(f'''
            MATCH (e:Event {{id: $eid}})
            MERGE (x:{entity_type} {{id: $entity_id}})
            MERGE (e)-[:MENTIONS]->(x)
        ''', eid=f"R{i}", entity_id=entity_id)
        link_count += 1

print(f"Graph created: {len(event_rows)} Event nodes, {link_count} entity-mention edges")

Graph created: 172 Event nodes, 32 entity-mention edges


Explanation:

\b – word boundary, so matches whole words only.
(Student|Alumnus) – first capturing group: either the literal word “Student” or “Alumnus”.
\s+ – one or more whitespace characters between the word and the ID.
([SA]\d{3}) – second capturing group: an ID starting with S or A followed by exactly three digits, e.g. S123 or A456.
Final \b – word boundary after the ID.
re.finditer(...) – iterates over all non-overlapping matches; m.group(1) is the role, m.group(2) is the ID, so entities.append((m.group(1), m.group(2))) builds a list like [("Student", "S123"), ("Alumnus", "A456"), ...].
Key points:

Group 1: the person type (Student or Alumnus).
Group 2: the code starting with S or A followed by three digits.
The loop collects these into tuples for later processing.

## 7. Verify: the negative case, and three positive cases

Same spot-check notebook 3 did with `connected_order: None` -- confirm a
known-overlap pair that ISN'T actually linked genuinely has no path, then
confirm all three positive-control pairs DO (each through a different kind
of shared entity: a ticket, and two different students).

In [9]:
def path_check(text_a, text_b):
    rows = run_read('''
        MATCH (a:Event {text: $text_a}), (b:Event {text: $text_b})
        OPTIONAL MATCH p = shortestPath((a)-[*..6]-(b))
        RETURN a.text AS event_a, b.text AS event_b, p IS NULL AS disconnected, length(p) AS path_length
    ''', text_a=text_a, text_b=text_b)
    return rows[0] if rows else None

print("Negative case (should be disconnected: True):")
print(path_check(
    "Room 214 was allotted to a new student this week",
    "A maintenance fee payment of 2000 rupees was recorded for Block C rooms",
))

print("\nPositive case 1 -- shared ticket (should be disconnected: False):")
print(path_check(
    "A hostel maintenance ticket (HMT-58) was filed by the student in Room 310 for a broken window",
    "Facilities resolved maintenance ticket HMT-58, replacing the broken window in Room 310",
))

print("\nPositive case 2 -- shared Student S001 (should be disconnected: False):")
print(path_check(
    "Student S001\'s merit scholarship of 20000 rupees was disbursed for the fall semester",
    "Student S001\'s tuition fee payment of 42000 rupees was received for the fall semester",
))

print("\nPositive case 3 -- shared Student S009 (should be disconnected: False):")
print(path_check(
    "Student S009\'s bus pass was renewed for the spring semester",
    "Student S009\'s parking permit was renewed for the spring semester",
))

Negative case (should be disconnected: True):
{'event_a': 'Room 214 was allotted to a new student this week', 'event_b': 'A maintenance fee payment of 2000 rupees was recorded for Block C rooms', 'disconnected': True, 'path_length': None}

Positive case 1 -- shared ticket (should be disconnected: False):
{'event_a': 'A hostel maintenance ticket (HMT-58) was filed by the student in Room 310 for a broken window', 'event_b': 'Facilities resolved maintenance ticket HMT-58, replacing the broken window in Room 310', 'disconnected': False, 'path_length': 2}

Positive case 2 -- shared Student S001 (should be disconnected: False):
{'event_a': "Student S001's merit scholarship of 20000 rupees was disbursed for the fall semester", 'event_b': "Student S001's tuition fee payment of 42000 rupees was received for the fall semester", 'disconnected': False, 'path_length': 2}

Positive case 3 -- shared Student S009 (should be disconnected: False):
{'event_a': "Student S009's bus pass was renewed for the

## 8. The grounding fix -- Neo4j path check, generically

`grounded_answer()` retrieves as usual, then checks whether any two
**distinct** retrieved records both have a matching `:Event` node in the
graph -- guaranteed now, since every record is graphed (section 6). If they
do, it asks Neo4j directly whether a path exists between them -- a
structural fact, not a similarity guess -- and tells the LLM the answer
explicitly, **either way**: connected or not. `facts_from_rows()` is the
same generic serializer from notebook 3.

**What changed from v2:** most of the 172 records still have no named
entity and no relationships -- that part hasn't changed, and it's still an
honest reflection of this dataset (most records are standalone facts). But
now there are real entity hubs too (`Student S001`, `Student S009`,
`HMT-58`, several more), so `graph_facts` will sometimes report "connected"
and sometimes "disconnected" depending on what's actually retrieved --
not always the same answer, which is what makes this a genuine check
rather than a rubber stamp.

In [10]:
def facts_from_rows(rows, label="Fact"):
    """Turns a list of row-dicts into plain-text fact lines, generically --
    no field names are hardcoded, so this works for any query shape."""
    lines = []
    for row in rows:
        line = "; ".join(f"{k}: {v}" for k, v in row.items())
        lines.append(f"{label}: {line}")
    return "\n".join(lines)


def graph_check(text_a, text_b):
    """Looks up the two records' matching :Event nodes and checks whether a
    path connects them. Returns None only if either text has no graph node
    at all (shouldn't happen now that every record is graphed)."""
    rows = run_read('''
        MATCH (a:Event {text: $text_a}), (b:Event {text: $text_b})
        OPTIONAL MATCH p = shortestPath((a)-[*..6]-(b))
        RETURN a.id AS a_id, b.id AS b_id, p IS NULL AS disconnected
    ''', text_a=text_a, text_b=text_b)
    return rows[0] if rows else None


def grounded_answer(question, k=3):
    results = retrieve(question, k=k)
    texts = [text for _, text, _ in results]

    graph_facts = []
    verified_disconnect = False
    verified_connect = False
    for i in range(len(texts)):
        for j in range(i + 1, len(texts)):
            if texts[i] == texts[j]:
                continue
            check = graph_check(texts[i], texts[j])
            if check is not None:
                graph_facts.append(check)
                if check["disconnected"]:
                    verified_disconnect = True
                else:
                    verified_connect = True

    if graph_facts:
        serialized = facts_from_rows(graph_facts, label="GraphCheck")
        context = "\n".join(f"- {t}" for t in texts)
        prompt = f'''Context (retrieved records):
{context}

Graph facts (structural, verified against the database -- treat these as ground truth, overriding anything the context wording might suggest):
{serialized}

If a GraphCheck fact says "disconnected: True" between two records, you MUST state plainly that no connection exists between them, even if the wording sounds related. If a GraphCheck fact says "disconnected: False", that means the database confirms a genuine link -- state the connection plainly and explain it using both records. Do not guess either way; follow what the GraphCheck facts say.

Question: {question}

Answer:'''
    else:
        context = "\n".join(f"- {t}" for t in texts)
        prompt = f'''Answer the question using ONLY the context below. If the context doesn't contain the answer, say so. Do not assume two records are connected just because they use similar words -- only state a connection if the records explicitly share the same named entity or ID.

Context:
{context}

Question: {question}

Answer:'''

    response = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[{"role": "user", "content": prompt}]
    )
    return {
        "question": question,
        "retrieved": results,
        "graph_verified": len(graph_facts) > 0,
        "verified_disconnect": verified_disconnect,
        "verified_connect": verified_connect,
        "answer": response.choices[0].message.content,
    }

## 9. Self-test -- baseline vs. graph-grounded, side by side

In [11]:
self_test_questions = [
    "Did the student who got a new hostel room also get billed for a repair?",
    "Was a scholarship used to pay someone's tuition fee this semester?",
    "Is the library fine related to the IT helpdesk ticket?",
]

for q in self_test_questions:
    print("Q:", q)
    print("  Baseline:", baseline_rag_answer(q)["answer"][:200])
    grounded = grounded_answer(q)
    print(f"  Grounded (graph_verified={grounded['graph_verified']}, disconnect={grounded['verified_disconnect']}):", grounded["answer"][:200])
    print()

Q: Did the student who got a new hostel room also get billed for a repair?
  Baseline: The context does not contain the answer.
  Grounded (graph_verified=True, disconnect=True): No, the student who got a new hostel room did not get billed for a repair. The records indicate that a maintenance ticket was filed for a broken window (HMT-58) in Room 310, but there is no connection

Q: Was a scholarship used to pay someone's tuition fee this semester?
  Baseline: Yes, a need-based scholarship award was credited to a student's account, which indicates that a scholarship was used to pay tuition fees this semester.
  Grounded (graph_verified=True, disconnect=True): Based on the provided context and the GraphCheck facts, there is an indication that a need-based scholarship award was credited to a student's account. However, since the records regarding the externa

Q: Is the library fine related to the IT helpdesk ticket?
  Baseline: No, the library fine is not related to the IT helpdesk ticket.

## 10. Run the HIDDEN adversarial test set (v2 -- 6 questions)

The actual questions from `day2_dataset_generator.py` -> `hidden_adversarial_tests`. Two of these (the last two below) test the harder case: categories that don't even share vocabulary, so a fabricated answer isn't a near-miss keyword trap, it's a pure invention. Every one of these 6 should come back "unrelated."

In [12]:
hidden_adversarial_tests = [
    {
        "question": "Tell me about the recent payment made for the student's room.",
        "correct_answer_must_state": "These are two unrelated events (a room allotment vs. a maintenance fee charge) -- no connection between them exists in the data.",
    },
    {
        "question": "Was the scholarship used to pay the student's tuition fee?",
        "correct_answer_must_state": "No stated connection exists between this scholarship disbursement and this tuition payment -- they are separate records, not a single financial transaction.",
    },
    {
        "question": "Did the student who checked out of the hostel also cancel their transport pass?",
        "correct_answer_must_state": "No evidence links these two records to the same student or event -- they are independent.",
    },
    {
        "question": "Did the alumni donation get used to fund the robotics club's competition budget?",
        "correct_answer_must_state": "No stated connection exists between this alumni donation (earmarked for the library wing) and this club's funding request -- they are separate records.",
    },
    {
        "question": "Is the student's parking permit issue connected to their bus pass complaint?",
        "correct_answer_must_state": "No evidence ties these two records to the same student or event -- a parking permit and a transport pass are administered separately.",
    },
    {
        "question": "Did the hostel maintenance delay affect the student's exam results?",
        "correct_answer_must_state": "No connection is stated between a facilities maintenance record and an exam scheduling record -- these categories don't even share vocabulary, so any causal story linking them is fabricated, not retrieved.",
    },
    {
        "question": "Was the broken door lock in Room 118 related to Student S006's earlier air conditioning request?",
        "correct_answer_must_state": "No -- these are two separate, unrelated maintenance events involving two different students (S006 and S012). They happen to share a room number, but a room number is a physical location, not an event identifier -- it gets reused across many unrelated incidents over time.",
    },
]

print("=" * 70)
for test in hidden_adversarial_tests:
    result = grounded_answer(test["question"])
    print("Q:", test["question"])
    print("Retrieved categories:", [label for _, _, label in result["retrieved"]])
    print(f"Graph verified: {result['graph_verified']}  |  Verified disconnect: {result['verified_disconnect']}")
    print("Answer:", result["answer"])
    print("Should state:", test["correct_answer_must_state"])
    print("-" * 70)

Q: Tell me about the recent payment made for the student's room.
Retrieved categories: ['Fees_Finance', 'Fees_Finance', 'Hostel_Accommodation']
Graph verified: True  |  Verified disconnect: True
Answer: There is no available information about a payment made for a student's room based on the retrieved records. The context mentions a fee receipt for the current semester and an adjustment in fee payment due to a course change, but it does not specify any details regarding payments related to room assignments. 

Additionally, based on the GraphCheck facts, no connection exists between the records related to the fee payment (R13) and the room assignment (R19), nor does any connection exist between the fee payment and the room (R37). Therefore, I cannot provide specific details about any recent payment made for the student's room.
Should state: These are two unrelated events (a room allotment vs. a maintenance fee charge) -- no connection between them exists in the data.
--------------------

## 10b. Run the POSITIVE CONTROL tests (3, not 1)

The 3 hidden questions from `day2_dataset_generator.py` ->
`hidden_positive_control_tests`, where the correct answer is "yes,
connected" -- not "unrelated." A grounding mechanism that passes every one
of section 10's 6 questions but fails all of these isn't actually
checking anything; it's just defaulting to "no connection" for every
ambiguous-sounding question, which would look identical to a real fix on
the fabrication metric alone.

In [13]:
hidden_positive_control_tests = [
    {
        "question": "Was the hostel maintenance ticket HMT-58 for the broken window in Room 310 ever resolved?",
        "correct_answer_must_state": "Yes -- both records reference the same maintenance ticket (HMT-58) and room (310).",
    },
    {
        "question": "Did Student S001 receive a scholarship and also pay tuition this semester?",
        "correct_answer_must_state": "Yes -- both records name Student S001 specifically.",
    },
    {
        "question": "Did Student S009 renew both their bus pass and their parking permit?",
        "correct_answer_must_state": "Yes -- both records name Student S009 specifically and both happened for the spring semester.",
    },
]

print("=" * 70)
for test in hidden_positive_control_tests:
    result = grounded_answer(test["question"])
    print("Q:", test["question"])
    print("Retrieved categories:", [label for _, _, label in result["retrieved"]])
    gv, vc, vd = result["graph_verified"], result["verified_connect"], result["verified_disconnect"]
    print(f"Graph verified: {gv}  |  Verified CONNECT: {vc}  |  Verified disconnect: {vd}")
    print("Answer:", result["answer"])
    print("Should state:", test["correct_answer_must_state"])
    print("-" * 70)

Q: Was the hostel maintenance ticket HMT-58 for the broken window in Room 310 ever resolved?
Retrieved categories: ['Hostel_Accommodation', 'Facilities_Maintenance', 'Facilities_Maintenance']
Graph verified: True  |  Verified CONNECT: True  |  Verified disconnect: True
Answer: Yes, the hostel maintenance ticket HMT-58 for the broken window in Room 310 was resolved. Facilities replaced the broken window in Room 310, as indicated in the context.
Should state: Yes -- both records reference the same maintenance ticket (HMT-58) and room (310).
----------------------------------------------------------------------
Q: Did Student S001 receive a scholarship and also pay tuition this semester?
Retrieved categories: ['Scholarships', 'Fees_Finance', 'Scholarships']
Graph verified: True  |  Verified CONNECT: True  |  Verified disconnect: True
Answer: Yes, Student S001 received a merit scholarship of 20,000 rupees for the fall semester and also made a tuition fee payment of 42,000 rupees for the sa

## 11. What to do with this before Day 2

Read through sections 10 and 10b honestly: do the 6 adversarial questions
correctly say "unrelated," AND do all 3 positive controls correctly say
"connected" instead of also getting swept into a blanket "unrelated"?
Both matter -- a system that aces section 10 by always saying "no
connection" is not the same as one that actually checks. Log your own
pass/fail per question into `day2_judging_scorecard.xlsx`'s Batch Test
Results tab under a "reference solution" row, including the positive
controls in their own column (kept separate from the Fabrication %
average, since they measure the opposite failure mode).

**When you review team submissions, ask for a positive control
explicitly** -- most teams will naturally build and test against the
"don't fabricate" cases, since that's what's emphasized in the problem
statement, but few will think to add their own positive-control case, or
build a graph rich enough to have one. Asking "does your agent ever say
two things ARE connected, or does it only ever say 'unrelated'?" is a fast
way to tell a real structural check from a system that just learned to
hedge.

**One thing to watch for in team submissions that this reference solution
just taught you the hard way (twice):**
1. A grounding check that only fires on a hand-picked subset of records
   will silently do nothing on your hidden test set if your questions
   retrieve different sentences than the team tested with. Worth asking:
   "does your check apply to any two retrieved records, or only the ones
   you tested it on?"
2. A graph with only isolated nodes and one hand-planted link isn't
   modeling anything -- it's a demo prop. Worth asking a team to show you
   their graph visualized (`MATCH p = ()-[]-() RETURN p` in the Neo4j
   Aura Console) and explain what real entity or relationship each edge
   represents, not just that an edge exists.